In [ ]:
import torch
import torch.nn as nn

## Implement Partial Cross Entropy Loss

In [3]:
class PCELoss(nn.Module):
    """
    Partial Cross Entropy Loss 
        - evaluates only at labeled points.
    """

    def __init__(self, ignore_index=-1, class_weights=None):
        super().__init__()

        self.ignore_index = ignore_index
        if class_weights is not None:
            self.class_weights = torch.as_tensor(class_weights, dtype=torch.float32)

    def forward(self, logits, point_labels):
        if logits.ndim != 4 or point_labels.ndim != 3:
            raise ValueError("Excepted logints [B, C, H, W] and labels [B, H, W]")

        return nn.functional.cross_entropy(logits,
                                           point_labels.long(),
                                           weight=self.class_weights,
                                           ignore_index=self.ignore_index,
                                           )
        


In [4]:
def make_point_labels(full_mask, fraction, ignore_index=-1):
    device = full_mask.device
    g = torch.Generator(device=device)

    points = torch.full_like(full_mask, ignore_index)
    B, H, W = full_mask.shape

    for b in range(B):
        valid = torch.ones(H, W, dtype=torch.bool, device=device)
        coords = torch.nonzero(valid, as_tuple=False)
        n = max(1, int(round(coords.shape[0] * fraction)))
        idx = torch.randperm(coords.shape[0], generator=g, device=device)[:n]
        selected = coords[idx]
        points[b, selected[:, 0], selected[:, 1]] = full_mask[b, selected[:, 0], selected[:, 1]]
    return points

In [5]:
import numpy as np

def confusion_matrix(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    valid = (y_true >= 0) & (y_true < num_classes)
    np.add.at(cm, (y_true[valid], y_pred[valid]), 1)
    return cm

def segmentation_metrics(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, num_classes)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp

    iou = tp / np.maximum(tp + fp + fn, 1)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / np.maximum(tp + fn, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)

    return {
        "mIoU": float(np.mean(iou)),
        "pixel_accuracy": float(tp.sum() / max(cm.sum(), 1)),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "per_class_iou": iou.tolist(),
    }
